# Credit Risk Scoring Project - Decision Trees & Ensemble Learning

## Data Cleaning & Preparation
- Downloading datasets
- Re-encoding categorical variables
- Doing the train/validation/test split

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

#%matplotlib inline

In [2]:
# !wget "https://raw.githubusercontent.com/gastonstat/CreditScoring/refs/heads/master/CreditScoring.csv"

df = pd.read_csv('CreditScoring.csv')      # na_values= ['0', '99999999']   # 0 and 99999999 were coded as na in the data manual
df.columns = df.columns.str.lower()
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


In [3]:
df.isnull().sum()

status       0
seniority    0
home         0
time         0
age          0
marital      0
records      0
job          0
expenses     0
income       0
assets       0
debt         0
amount       0
price        0
dtype: int64

## Categorical Variables

In [4]:
# status
df.status.unique()
print(df.status.value_counts())
df.status = df.status.map({1:'good', 2:'default', 0:'unknown'}) #unknown
print(df.status.value_counts())

# Home
df.home = df.home.map({1:'rent', 2:'owner', 3:'private', 4:'ignore', 5:'parents', 6:'other', 0:'unknown'}) #unknown
print(df.home.value_counts())

#Marital
df.marital = df.marital.map({1:'single', 2:'married', 3:'widow', 4:'separated', 5:'divorced', 0:'unknown'}) #unknown
print(df.marital.value_counts())

# Records 
print(df.records.value_counts())
df.records.unique()
df.records = df.records.map({1:'no', 2:'yes'})
print(df.records.value_counts())

# Job 
df.job.nunique()
print(df.job.value_counts())
df.job = df.job.map({1:'fixed', 2:'partime', 3:'freelance', 4:'others', 0:'unknown'}) #unknown
print(df.job.value_counts())

status
1    3200
2    1254
0       1
Name: count, dtype: int64
status
good       3200
default    1254
unknown       1
Name: count, dtype: int64
home
owner      2107
rent        973
parents     783
other       319
private     247
ignore       20
unknown       6
Name: count, dtype: int64
marital
married      3241
single        978
separated     130
widow          67
divorced       38
unknown         1
Name: count, dtype: int64
records
1    3682
2     773
Name: count, dtype: int64
records
no     3682
yes     773
Name: count, dtype: int64
job
1    2806
3    1024
2     452
4     171
0       2
Name: count, dtype: int64
job
fixed        2806
freelance    1024
partime       452
others        171
unknown         2
Name: count, dtype: int64


In [5]:
#dtypes after checking categorical
df.dtypes

status       object
seniority     int64
home         object
time          int64
age           int64
marital      object
records      object
job          object
expenses      int64
income        int64
assets        int64
debt          int64
amount        int64
price         int64
dtype: object

In [6]:
# Numeric variables
# # how many missing values (coded as 99999999) 
df.describe().round()

# We can see the missing codes(99999999) in the variables -> assets, income, debts, 
# lets replace those with nan from np

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,763317.0,1060341.0,404382.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,8703625.0,10217569.0,6344253.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3500.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,166.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,99999999.0,99999999.0,99999999.0,5000.0,11140.0


In [7]:
#Replace all cells containing 99999999 with nan
miss_columns = ['assets', 'income', 'debt']


for col in miss_columns:
    df[col] = df[col].replace(to_replace = 99999999.0 , value = np.nan)  
    
#or 

# df[miss_columns] = df[miss_columns].replace(to_replace=99999999.0, value = np.nan)

df.shape

(4455, 14)

In [8]:
df.isnull().sum()  #only coded 999999 as missing, he left 0

status        0
seniority     0
home          0
time          0
age           0
marital       0
records       0
job           0
expenses      0
income       34
assets       47
debt         18
amount        0
price         0
dtype: int64

In [9]:
# remove unknown rows from status variable and reset index 
df = df[df.status != 'unknown'].reset_index(drop=True)
df.shape

(4454, 14)

In [10]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4454.0,4454.0,4454.0,4454.0,4420.0,4407.0,4436.0,4454.0,4454.0
mean,8.0,46.0,37.0,56.0,131.0,5404.0,343.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,86.0,11574.0,1246.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1117.0
50%,5.0,48.0,36.0,51.0,120.0,3000.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,165.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,959.0,300000.0,30000.0,5000.0,11140.0


## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

df_full_train, df_val = train_test_split(df, test_size=0.2, random_state=11)  # we get val 20%

df_train, df_test = train_test_split(df_full_train, test_size=0.20, random_state=11) # we get train & test here

In [21]:
print(df_full_train.shape)
print(f'train set is {df_train.shape}')
print('validation set is ', df_val.shape) # 25%
print(f'test set is {df_test.shape}') #20

(3563, 14)
train set is (2850, 14)
validation set is  (891, 14)
test set is (713, 14)


In [14]:
#reset the indices

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [16]:
# Prep target outcome in train, val, test

#flag default rows as 1, good as 0
y_train = (df_train.status == 'default').astype('int')   #(df_full_train.status == 'good').astype('int')  --> flags good as 1, default as 0
y_val = (df_val.status == 'default').astype('int')
y_test = (df_test.status =='default').astype('int')

print(y_train.value_counts())
print(y_val.value_counts())
print(y_test.value_counts())

status
0    1921
1     751
Name: count, dtype: int64
status
0    793
1    321
Name: count, dtype: int64
status
0    486
1    182
Name: count, dtype: int64


In [ ]:
#remove target outcome from partitions to get X data

del df_train['status']
del df_val['status']
del df_test['status']

df_train.shape
df_val.shape
df_test.shape

(668, 13)

In [19]:
df_train

,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,0,owner,48,24,married,no,partime,35,150.0,5000.0,3130.0,1000,1404
1,2,parents,60,48,married,yes,fixed,45,106.0,1500.0,0.0,1000,1253
2,1,parents,12,53,single,no,fixed,35,160.0,0.0,0.0,250,1456
3,6,owner,48,35,married,no,freelance,45,200.0,7000.0,0.0,1540,1603
4,1,parents,48,40,married,no,fixed,75,121.0,0.0,0.0,1320,1600
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2667,20,owner,36,42,married,no,freelance,90,250.0,4500.0,0.0,1500,1750
2668,6,other,36,35,married,no,freelance,35,23.0,3000.0,0.0,250,350
2669,21,rent,36,42,married,no,fixed,74,135.0,450.0,0.0,240,1275
2670,19,owner,36,47,married,no,fixed,35,110.0,12000.0,1600.0,1400,1883


# 6.3 Decision Trees
 - How a Decision Tree Looks Like
 - Training a Decision Tree
 - Controlling the size of a tree

In [18]:
df_train.columns

Index(['seniority', 'home', 'time', 'age', 'marital', 'records', 'job',
       'expenses', 'income', 'assets', 'debt', 'amount', 'price'],
      dtype='object')

# 6.4 Decision Tree Learning Algorithm
- Finding the best spit for one column
- Finding the best Split for the entire dataset
- Stopping criteria
- Decision Tree Algorithm


# 6.5 Decision Tree Parameter Tuning
- Selecting max_depth
- Selecting min_samples_leaf

# 6.6 Ensembles and Random Forest
- Board of experts
- Ensembling models
- Random Forest - ensembling decision trees
- Tuning Random Forest

## Other Useful Parameters

# 6.7 Gradient Boosting and XGBoost

- Gradient Boosting vs Random Forest
- Installing XGBoost
- Training the first model
- Performance Monitoring
- Parsing xgboost's monitoring output